# Visor interactivo de los indicadores EMICRON

Este notebook conserva la estructura del visor SDMX-like de GEIH: audita las dimensiones, permite escoger geografía y segmentación y presenta, junto al gráfico, la nota metodológica, el universo, las variables, las preguntas fuente, la fórmula y la procedencia del indicador.

El visor consulta el Parquet publicado; no recalcula ni modifica indicadores.

## 1. Dependencias y fuentes de metadatos

Además del Parquet, el visor lee `VARIABLE_MAP` e `INDICATORS` del catálogo Excel. Para recuperar el texto de las preguntas consulta el inventario de diccionarios EMICRON.

In [1]:
from pathlib import Path
import html
import json

import duckdb
import ipywidgets as widgets
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import pandas as pd
from IPython.display import HTML, clear_output, display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 200)

repo = Path.cwd()
if not (repo / "outputs").exists() and (repo.parent / "outputs").exists():
    repo = repo.parent

PARQUET_PATH = repo / "outputs" / "emicron_sdmx" / "EMICRON_indicadores_SDMX.parquet"
INVENTORY_PATH = repo / "outputs" / "emicron_sdmx" / "EMICRON_indicadores_SDMX.xlsx"
QUESTION_PATH = repo.parent / "EMICRON" / "diccionarios" / "geih_questions_options.csv"

for required_path in (PARQUET_PATH, INVENTORY_PATH, QUESTION_PATH):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

VARIABLE_MAP = pd.read_excel(INVENTORY_PATH, sheet_name="VARIABLE_MAP", dtype=object)
INDICATOR_CATALOG = pd.read_excel(INVENTORY_PATH, sheet_name="INDICATORS", dtype=object)
QUESTIONS_RAW = pd.read_csv(QUESTION_PATH, dtype=object, low_memory=False)

question_columns = ["year", "variable_code", "question_text", "variable_url"]
QUESTIONS = (
    QUESTIONS_RAW[question_columns]
    .dropna(subset=["variable_code", "question_text"])
    .drop_duplicates()
    .copy()
)
QUESTIONS["year"] = pd.to_numeric(QUESTIONS["year"], errors="coerce").astype("Int64")

con = duckdb.connect()
print(f"Parquet: {PARQUET_PATH.resolve()}")
print(f"Catálogo: {INVENTORY_PATH.resolve()}")
print(f"Preguntas: {QUESTION_PATH.resolve()}")

Parquet: C:\Users\jorti\OneDrive\Escritorio\agregador\outputs\emicron_sdmx\EMICRON_indicadores_SDMX.parquet
Catálogo: C:\Users\jorti\OneDrive\Escritorio\agregador\outputs\emicron_sdmx\EMICRON_indicadores_SDMX.xlsx
Preguntas: C:\Users\jorti\OneDrive\Escritorio\EMICRON\diccionarios\geih_questions_options.csv


## 2. Auditoría de dimensiones disponibles

Una dimensión con un único valor `_T` sólo contiene el total. Esta tabla permite verificar qué desagregaciones existen realmente antes de graficar.

In [2]:
DIMENSIONS = [
    "CATEGORY", "AREA", "DOMAIN", "CLASE", "URBAN_RURAL",
    "SEX", "HEAD_SEX", "AGE", "REF_AREA", "GEO_LEVEL",
    "DEPT_CODE", "MUNI_CODE",
]

dimension_audit = pd.DataFrame([
    {
        "DIMENSION": dimension,
        "VALUES": con.execute(
            f"SELECT COUNT(DISTINCT {dimension}) FROM read_parquet(?)",
            [PARQUET_PATH.as_posix()],
        ).fetchone()[0],
        "CODES": ", ".join(
            row[0]
            for row in con.execute(
                f"SELECT DISTINCT CAST({dimension} AS VARCHAR) FROM read_parquet(?) ORDER BY 1 LIMIT 25",
                [PARQUET_PATH.as_posix()],
            ).fetchall()
            if row[0] is not None
        ),
    }
    for dimension in DIMENSIONS
])

dimension_audit

,DIMENSION,VALUES,CODES
0,CATEGORY,194,"01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 10_MAS, 11, 12, 1_A_MENOS_3, 3_A_MENOS_5, 5_A_MENOS_10, ACTIVITY_01, ACTIVITY_02, ACTIVITY_03, ACTIVITY_04, ACTIVITY_05, ACTIVITY_06, AC..."
1,AREA,25,"05, 08, 11, 13, 15, 17, 18, 19, 20, 23, 27, 41, 44, 47, 50, 52, 54, 63, 66, 68, 70, 73, 76, 88, _T"
2,DOMAIN,1,_T
3,CLASE,3,"1, 2, _T"
4,URBAN_RURAL,3,"R, U, _T"
5,SEX,3,"F, M, _T"
6,HEAD_SEX,1,_T
7,AGE,1,_T
8,REF_AREA,26,"CO, CO-05, CO-08, CO-11, CO-13, CO-15, CO-17, CO-18, CO-19, CO-20, CO-23, CO-25, CO-27, CO-41, CO-44, CO-47, CO-50, CO-52, CO-54, CO-63, CO-66, CO-68, CO-70, CO-73, CO-76"
9,GEO_LEVEL,5,"AREA, CLASS, DEP, DEP_CLASS, NAT"


## 3. Visor

1. Selecciona el indicador.
2. Escoge nivel geográfico y territorio.
3. Deja `Líneas por` en `Automático` o selecciona una dimensión.
4. Ajusta categoría y sexo del propietario/líder.
5. Pulsa **Graficar**.

Después de la tabla de resultados se presentan automáticamente los metadatos, preguntas y fórmula usados para el cálculo.

In [3]:
DIMENSION_LABELS = {
    "AUTO": "Automático",
    "CATEGORY": "Categoría del indicador",
    "AREA": "Área",
    "DOMAIN": "Dominio",
    "CLASE": "Clase",
    "URBAN_RURAL": "Urbano/rural",
    "SEX": "Sexo del propietario/líder",
    "HEAD_SEX": "Sexo de la jefatura",
    "AGE": "Edad/grupo de edad",
    "DEPT_CODE": "Departamento",
}

VALUE_LABELS = {
    "_T": "Total", "M": "Hombre", "F": "Mujer", "CO": "Colombia",
    "NAT": "Nacional", "AREA": "Área", "CLASS": "Clase urbano/rural",
    "DEP": "Departamento", "DEP_CLASS": "Departamento por clase",
    "U": "Cabecera", "R": "Centros poblados y rural disperso",
    "1": "Cabecera", "2": "Centros poblados y rural disperso",
    "00": "Total nacional", "0000": "Total nacional",
}

AREA_LABELS = {
    "05": "Medellín - Valle de Aburrá", "08": "Barranquilla - Soledad",
    "11": "Bogotá", "13": "Cartagena", "15": "Tunja",
    "17": "Manizales - Villamaría", "18": "Florencia", "19": "Popayán",
    "20": "Valledupar", "23": "Montería", "27": "Quibdó",
    "41": "Neiva", "44": "Riohacha", "47": "Santa Marta",
    "50": "Villavicencio", "52": "Pasto",
    "54": "Cúcuta - Villa del Rosario - Los Patios - El Zulia",
    "63": "Armenia", "66": "Pereira - Dosquebradas - La Virginia",
    "68": "Bucaramanga - Floridablanca - Girón - Piedecuesta",
    "70": "Sincelejo", "73": "Ibagué", "76": "Cali - Yumbo",
}

DEPARTMENT_LABELS = {
    "05": "Antioquia", "08": "Atlántico", "11": "Bogotá D.C.",
    "13": "Bolívar", "15": "Boyacá", "17": "Caldas", "18": "Caquetá",
    "19": "Cauca", "20": "Cesar", "23": "Córdoba", "25": "Cundinamarca",
    "27": "Chocó", "41": "Huila", "44": "La Guajira", "47": "Magdalena",
    "50": "Meta", "52": "Nariño", "54": "Norte de Santander",
    "63": "Quindío", "66": "Risaralda", "68": "Santander", "70": "Sucre",
    "73": "Tolima", "76": "Valle del Cauca",
}


def geography_code(code):
    text = str(code).strip()
    return text.zfill(2) if text.isdigit() else text


def area_label(code):
    code = geography_code(code)
    return AREA_LABELS.get(code, f"Área {code}")


def department_label(code):
    code = geography_code(code)
    return DEPARTMENT_LABELS.get(code, f"Departamento {code}")


def display_dimension_label(dimension, code):
    if dimension == "AREA":
        return area_label(code)
    if dimension == "DEPT_CODE":
        return department_label(code)
    return VALUE_LABELS.get(str(code), str(code))


def format_display_number(value):
    if pd.isna(value):
        return "—"
    formatted = f"{float(value):,.2f}"
    return formatted.replace(",", "X").replace(".", ",").replace("X", ".")


def format_observation_value(value, unit):
    formatted = format_display_number(value)
    return f"{formatted} %" if str(unit).upper() == "PERCENT" and formatted != "—" else formatted


def value_axis_label(unit):
    labels = {"PERCENT": "Porcentaje (%)", "THOUSAND_COP": "Miles de pesos"}
    return labels.get(str(unit).upper(), "Valor")


def pretty_rule(value):
    text = str(value).strip()
    payload = text.removeprefix("RULE_JSON:")
    try:
        return json.dumps(json.loads(payload), ensure_ascii=False, indent=2)
    except (json.JSONDecodeError, TypeError):
        return text


catalog = con.execute("""
SELECT INDICATOR, ANY_VALUE(INDICATOR_NAME) INDICATOR_NAME,
       ANY_VALUE(THEME) THEME, MIN(SOURCE_ROW) SOURCE_ROW
FROM read_parquet(?)
GROUP BY INDICATOR
ORDER BY SOURCE_ROW
""", [PARQUET_PATH.as_posix()]).fetchdf()

indicator_labels = {
    f"{row.INDICATOR} — {row.INDICATOR_NAME}": row.INDICATOR
    for row in catalog.itertuples(index=False)
}

indicator_widget = widgets.Combobox(
    options=list(indicator_labels), value=next(iter(indicator_labels)),
    description="Indicador:", ensure_option=True, continuous_update=False,
    layout=widgets.Layout(width="900px"),
)
segment_widget = widgets.Dropdown(
    options=[(DIMENSION_LABELS["AUTO"], "AUTO")] + [
        (DIMENSION_LABELS[d], d)
        for d in ["CATEGORY", "AREA", "URBAN_RURAL", "SEX", "DEPT_CODE"]
    ],
    value="AUTO", description="Líneas por:", layout=widgets.Layout(width="420px"),
)
geography_level_widget = widgets.Dropdown(
    options=[("Nacional", "NAT"), ("Área", "AREA"),
             ("Clase urbano/rural", "CLASS"), ("Departamento", "DEP"),
             ("Departamento por clase", "DEP_CLASS")],
    value="NAT", description="Geografía:", layout=widgets.Layout(width="420px"),
)
geography_code_widget = widgets.Dropdown(
    options=[("Colombia", "CO")], value="CO", description="Territorio:",
    layout=widgets.Layout(width="420px"),
)
filter_widgets = {
    dimension: widgets.Dropdown(
        description=DIMENSION_LABELS[dimension] + ":",
        layout=widgets.Layout(width="410px"),
    )
    for dimension in ["CATEGORY", "SEX", "HEAD_SEX", "AGE"]
}
plot_button = widgets.Button(description="Graficar", button_style="primary", icon="line-chart")
viewer_output = widgets.Output()


def selected_indicator():
    raw = str(indicator_widget.value).strip()
    if raw in indicator_labels:
        return indicator_labels[raw]
    code = raw.split(" — ", 1)[0].strip().upper()
    if code in set(indicator_labels.values()):
        return code
    raise ValueError(f"Indicador no reconocido: {raw}")


def geography_options(indicator, geo_level):
    if geo_level == "NAT":
        return [("Colombia", "CO")]
    if geo_level == "DEP_CLASS":
        rows = con.execute("""
            SELECT DISTINCT CAST(DEPT_CODE AS VARCHAR), CAST(CLASE AS VARCHAR)
            FROM read_parquet(?) WHERE INDICATOR=? AND GEO_LEVEL='DEP_CLASS'
            ORDER BY 1,2
        """, [PARQUET_PATH.as_posix(), indicator]).fetchall()
        return [(f"{department_label(dept)} · {VALUE_LABELS.get(clase, clase)}", f"{dept}|{clase}")
                for dept, clase in rows]
    dimension = {"AREA": "AREA", "CLASS": "CLASE", "DEP": "DEPT_CODE"}[geo_level]
    rows = con.execute(f"""
        SELECT DISTINCT CAST({dimension} AS VARCHAR) code
        FROM read_parquet(?) WHERE INDICATOR=? AND GEO_LEVEL=?
        ORDER BY TRY_CAST(code AS INTEGER) NULLS LAST, code
    """, [PARQUET_PATH.as_posix(), indicator, geo_level]).fetchall()
    labeler = {"AREA": area_label, "CLASS": lambda x: VALUE_LABELS.get(x, x), "DEP": department_label}[geo_level]
    options = [(labeler(row[0]), row[0]) for row in rows]
    options.insert(0, ({"AREA": "Todas las áreas", "CLASS": "Todas las clases", "DEP": "Todos los departamentos"}[geo_level], "*"))
    return options


def geography_conditions(indicator):
    level = geography_level_widget.value
    code = geography_code_widget.value
    where, parameters = ["INDICATOR=?", "GEO_LEVEL=?"], [indicator, level]
    if code == "*":
        return where, parameters
    if level == "NAT":
        where.append("REF_AREA='CO'")
    elif level == "AREA":
        where.append("CAST(AREA AS VARCHAR)=?"); parameters.append(str(code))
    elif level == "CLASS":
        where.append("CAST(CLASE AS VARCHAR)=?"); parameters.append(str(code))
    elif level == "DEP":
        where.append("CAST(DEPT_CODE AS VARCHAR)=?"); parameters.append(str(code))
    elif level == "DEP_CLASS":
        department, clase = str(code).split("|", 1)
        where.extend(["CAST(DEPT_CODE AS VARCHAR)=?", "CAST(CLASE AS VARCHAR)=?"])
        parameters.extend([department, clase])
    return where, parameters


def dimension_values(indicator, dimension):
    where, parameters = geography_conditions(indicator)
    rows = con.execute(f"""
        SELECT DISTINCT CAST({dimension} AS VARCHAR) code
        FROM read_parquet(?) WHERE {' AND '.join(where)} AND {dimension} IS NOT NULL
        ORDER BY code
    """, [PARQUET_PATH.as_posix(), *parameters]).fetchall()
    return [row[0] for row in rows]


def automatic_segment(indicator):
    if geography_code_widget.value == "*":
        return {"AREA": "AREA", "CLASS": "URBAN_RURAL", "DEP": "DEPT_CODE"}.get(
            geography_level_widget.value, "CATEGORY"
        )
    for dimension in ["CATEGORY", "SEX", "HEAD_SEX", "AGE"]:
        if len(dimension_values(indicator, dimension)) > 1:
            return dimension
    return "CATEGORY"


def effective_segment():
    return automatic_segment(selected_indicator()) if segment_widget.value == "AUTO" else segment_widget.value


def refresh_geography(*_):
    options = geography_options(selected_indicator(), geography_level_widget.value)
    geography_code_widget.options = options
    values = [value for _, value in options]
    geography_code_widget.value = "CO" if "CO" in values else values[0]
    refresh_filters()


def refresh_filters(*_):
    indicator = selected_indicator()
    for dimension, widget in filter_widgets.items():
        values = dimension_values(indicator, dimension)
        if dimension == "CATEGORY":
            labels = dict(con.execute("""
                SELECT DISTINCT CAST(CATEGORY AS VARCHAR), CATEGORY_LABEL
                FROM read_parquet(?) WHERE INDICATOR=? ORDER BY 1
            """, [PARQUET_PATH.as_posix(), indicator]).fetchall())
            widget.options = [(labels.get(value, value), value) for value in values]
        else:
            widget.options = [(VALUE_LABELS.get(value, value), value) for value in values]
        if values:
            widget.value = "_T" if "_T" in values else values[0]
    segment = effective_segment()
    for dimension, widget in filter_widgets.items():
        widget.disabled = dimension == segment


def variable_map_for_indicator(indicator):
    columns = ["CONCEPT", "ROLE", "HARMONIZED_VARIABLE", "START_YEAR", "END_YEAR", "NOTE"]
    rows = VARIABLE_MAP.loc[VARIABLE_MAP["SOURCE_ITEM_ID"].astype(str).eq(indicator), columns]
    return rows.dropna(how="all").sort_values(["ROLE", "CONCEPT"], kind="stable")


def questions_for_indicator(indicator):
    mapping = variable_map_for_indicator(indicator)
    variables = sorted(set(mapping["HARMONIZED_VARIABLE"].dropna().astype(str)))
    rows = []
    for variable in variables:
        available = QUESTIONS.loc[QUESTIONS["variable_code"].astype(str).eq(variable)]
        if available.empty:
            rows.append({"VARIABLE": variable, "START_YEAR": pd.NA, "END_YEAR": pd.NA,
                         "QUESTION": "Variable derivada o sin pregunta directa; consultar la fórmula.", "URL": pd.NA})
            continue
        grouped = available.groupby(["variable_code", "question_text", "variable_url"], dropna=False)["year"].agg(["min", "max"]).reset_index()
        for row in grouped.itertuples(index=False):
            rows.append({"VARIABLE": variable, "START_YEAR": row.min, "END_YEAR": row.max,
                         "QUESTION": row.question_text, "URL": row.variable_url})
    return pd.DataFrame(rows, columns=["VARIABLE", "START_YEAR", "END_YEAR", "QUESTION", "URL"])


def catalog_metadata(indicator):
    columns = ["INDICATOR", "INDICATOR_NAME", "THEME", "AGGREGATION_TYPE", "UNIT",
               "START_YEAR", "END_YEAR", "WEIGHT_NATIONAL", "WEIGHT_DEPARTMENT",
               "METHODOLOGICAL_NOTE", "SOURCE_FILE", "SOURCE_SHEET", "SOURCE_URL"]
    return INDICATOR_CATALOG.loc[INDICATOR_CATALOG["INDICATOR"].astype(str).eq(indicator), columns]


def load_series(indicator, segment):
    where, geo_parameters = geography_conditions(indicator)
    parameters = [PARQUET_PATH.as_posix(), *geo_parameters]
    for dimension, widget in filter_widgets.items():
        if dimension == segment or widget.value is None:
            continue
        where.append(f"CAST({dimension} AS VARCHAR)=?")
        parameters.append(str(widget.value))
    category_label = "CATEGORY_LABEL" if segment == "CATEGORY" else f"CAST({segment} AS VARCHAR)"
    return con.execute(f"""
        SELECT TIME_PERIOD, OBS_VALUE, INDICATOR_NAME, UNIT, FREQ, OBS_STATUS,
               SOURCE_VARIABLES, FORMULA, METHODOLOGY_NOTE, UNIVERSE, SOURCE,
               WEIGHT_TYPE, ESTIMATION_SCOPE,
               CAST({segment} AS VARCHAR) SERIES_CODE, {category_label} SERIES_LABEL
        FROM read_parquet(?) WHERE {' AND '.join(where)}
        ORDER BY SERIES_CODE, TIME_PERIOD
    """, parameters).fetchdf()


def show_methodology(indicator, data):
    display(HTML("<h3>Metodología del indicador seleccionado</h3>"))
    display(catalog_metadata(indicator).reset_index(drop=True))
    notes = sorted({str(v).strip() for v in data["METHODOLOGY_NOTE"].dropna() if str(v).strip()})
    print("\nNota metodológica:")
    print("\n".join(f"• {value}" for value in notes) if notes else "• Sin nota adicional en el catálogo.")
    universes = sorted({str(v).strip() for v in data["UNIVERSE"].dropna() if str(v).strip()})
    print("\nUniverso o denominador:")
    for value in universes:
        print(pretty_rule(value))
    variables = sorted({str(v).strip() for v in data["SOURCE_VARIABLES"].dropna() if str(v).strip()})
    print("\nVariables utilizadas:")
    print("\n".join(f"• {value}" for value in variables))
    variable_details = variable_map_for_indicator(indicator)
    if not variable_details.empty:
        print("\nMapa de variables EMICRON:")
        display(variable_details.reset_index(drop=True))
    question_details = questions_for_indicator(indicator)
    print("\nPreguntas fuente utilizadas para el cálculo:")
    if question_details.empty:
        print("• No hay pregunta directa; el indicador usa una variable derivada.")
    else:
        display(question_details)
    formulas = sorted({str(v).strip() for v in data["FORMULA"].dropna() if str(v).strip()})
    print("\nFórmula ejecutada:")
    for value in formulas:
        display(HTML(f"<pre style='white-space:pre-wrap'>{html.escape(pretty_rule(value))}</pre>"))
    sources = sorted({str(v).strip() for v in data["SOURCE"].dropna() if str(v).strip()})
    print("Fuente de las observaciones: " + "; ".join(sources))


def plot_selected(*_):
    indicator = selected_indicator()
    segment = effective_segment()
    if geography_code_widget.value == "*":
        allowed = {"AREA": {"AREA"}, "CLASS": {"URBAN_RURAL"}, "DEP": {"DEPT_CODE"}}.get(geography_level_widget.value, set())
        if segment not in allowed:
            with viewer_output:
                clear_output(wait=True)
                print("Al elegir todos los territorios, usa la dimensión territorial correspondiente en 'Líneas por'.")
            return
    data = load_series(indicator, segment)
    with viewer_output:
        clear_output(wait=True)
        if data.empty:
            print("No existen observaciones con la combinación seleccionada.")
            return
        data["DATE"] = pd.to_datetime(data["TIME_PERIOD"].astype(str) + "-01-01")
        data["SERIES_LABEL"] = data.apply(
            lambda row: row["SERIES_LABEL"] if segment == "CATEGORY" else display_dimension_label(segment, row["SERIES_CODE"]), axis=1
        )
        number_of_series = data["SERIES_CODE"].nunique()
        selected_geo_label = next(label for label, value in geography_code_widget.options if value == geography_code_widget.value)
        print(f"Segmentación: {DIMENSION_LABELS[segment]}. Se grafican {number_of_series} líneas.")
        print(f"Geografía: {VALUE_LABELS.get(geography_level_widget.value, geography_level_widget.value)} · {selected_geo_label}.")
        print("Factor(es): " + ", ".join(sorted(data["WEIGHT_TYPE"].dropna().astype(str).unique())))
        fig, ax = plt.subplots(figsize=(14, 6.5))
        for (_, label), group in data.groupby(["SERIES_CODE", "SERIES_LABEL"], sort=True):
            ax.plot(group["DATE"], group["OBS_VALUE"], linewidth=1.7,
                    marker="o" if number_of_series <= 12 else None, markersize=3, label=str(label))
        title, unit = data.iloc[0]["INDICATOR_NAME"], data.iloc[0]["UNIT"]
        ax.set_title(f"{title}\n{selected_geo_label}", loc="left", fontsize=14, fontweight="bold")
        ax.set_xlabel("Año"); ax.set_ylabel(value_axis_label(unit))
        ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: format_observation_value(value, unit)))
        ax.grid(axis="y", alpha=0.25); ax.grid(axis="x", visible=False)
        ax.spines[["top", "right"]].set_visible(False)
        if number_of_series <= 40:
            ax.legend(title=DIMENSION_LABELS[segment], bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
        else:
            print("La leyenda se omite porque hay más de 40 series.")
        fig.tight_layout(); plt.show()
        preview = data[["TIME_PERIOD", "SERIES_CODE", "SERIES_LABEL", "OBS_VALUE", "UNIT", "WEIGHT_TYPE", "OBS_STATUS"]].head(50).copy()
        preview["OBS_VALUE"] = preview["OBS_VALUE"].map(lambda value: format_observation_value(value, unit))
        display(preview.rename(columns={"OBS_VALUE": "Valor"}))
        show_methodology(indicator, data)


indicator_widget.observe(refresh_geography, names="value")
segment_widget.observe(refresh_filters, names="value")
geography_level_widget.observe(refresh_geography, names="value")
geography_code_widget.observe(refresh_filters, names="value")
plot_button.on_click(plot_selected)
refresh_geography()

filter_grid = widgets.GridBox(
    children=list(filter_widgets.values()),
    layout=widgets.Layout(grid_template_columns="repeat(2, 430px)", grid_gap="6px 10px"),
)
display(widgets.VBox([
    indicator_widget,
    widgets.HBox([geography_level_widget, geography_code_widget]),
    segment_widget, filter_grid, plot_button,
]), viewer_output)
plot_selected()

Output()

## 4. Metodología del indicador seleccionado

Esta consulta independiente replica la última sección del visor GEIH y permite revisar los metadatos guardados físicamente en el Parquet.

In [ ]:
indicator = selected_indicator()
con.execute("""
SELECT DISTINCT INDICATOR, INDICATOR_NAME, UNIT, OBS_STATUS,
       SOURCE_VARIABLES, FORMULA, UNIVERSE, METHODOLOGY_NOTE, SOURCE
FROM read_parquet(?)
WHERE INDICATOR=?
""", [PARQUET_PATH.as_posix(), indicator]).fetchdf()